# A2 — extraction check on the saved 60-row sample (multi-GPU fix)
GPU T4 x2, Internet ON, HF secret ON. ~5-8 min. Reuses `artifacts/prerun_A1/gen_std_160.jsonl` (greedy answers from notebook A, same pinned revision).

In [ ]:
import os, shutil, glob, json, subprocess
hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/FROZEN_v2.json', recursive=True)]
assert hits, 'attach the rag-noise-probe dataset (with artifacts/data/FROZEN_v2.json)'
SRC = os.path.dirname(os.path.dirname(hits[0])); REPO = '/kaggle/working/rag-noise-probe'
shutil.copytree(SRC, REPO, dirs_exist_ok=True); os.chdir(REPO)
!pip install -q rank_bm25 pyarrow pytest 2>&1 | tail -1
!python -m pytest tests -q 2>&1 | tail -2
import transformers, torch; print('transformers', transformers.__version__, 'torch', torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
_tok, _label = None, None
for _l in ['HF_TOKEN', 'hf_token', 'HF_token', 'Hf_token', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'huggingface_token', 'HF']:
    try:
        _tok = UserSecretsClient().get_secret(_l); _label = _l; break
    except Exception as e:
        print('secret label', repr(_l), '-> not available', type(e).__name__)
assert _tok, 'No Hugging Face secret reachable from this run (attach it under Add-ons -> Secrets)'
print('using secret label:', repr(_label))   # the value is never printed
login(token=_tok); del _tok
SECRET_ONLY = os.environ.get('SECRET_ONLY') == '1'
MODEL = 'meta-llama/Llama-2-7b-chat-hf'          # revision is pinned in artifacts/data/model_pin.json
BENCH = 'artifacts/data/benchmark_v2.jsonl'
RUN = 'artifacts/runs/llama2-7b'; os.makedirs(RUN, exist_ok=True)

In [ ]:
!python -m src.extract.redeep_extract --model $MODEL --gen artifacts/prerun_A1/gen_std_160.jsonl --bench $BENCH --outdir $RUN/prerun2 --checkpoint_every 20 --cmp_set artifacts/data/comparison_set_v2.json
import pandas as pd, numpy as np
f = pd.read_parquet(f'{RUN}/prerun2/redeep_feats.parquet'); m = json.load(open(f'{RUN}/prerun2/redeep_meta.json'))
g = [json.loads(l) for l in open('artifacts/prerun_A1/gen_std_160.jsonl')]
E = np.concatenate([np.asarray(x) for x in f.ecs_mean]); P = np.stack(f.pks_mean.values)
print('rows', len(f), '| tf_match mean', f.tf_match.mean(), 'min', f.tf_match.min())
print('head_source', m['head_source'], '| revision', m['revision'], '| n_heads', len(m['head_list']))
print('ECS range', E.min().round(3), E.max().round(3), '| PKS mean by layer (0,8,16,24,31):', P.mean(0)[[0,8,16,24,31]].round(1))
print('non-finite:', int((~np.isfinite(E)).sum() + (~np.isfinite(P)).sum()))

In [ ]:
import zipfile
with zipfile.ZipFile('/kaggle/working/prerun2.zip','w',zipfile.ZIP_DEFLATED) as z:
    for dp,dn,fn in os.walk(f'{RUN}/prerun2'):
        if '.parts' in dp: continue
        for n in fn: p=os.path.join(dp,n); z.write(p,p.replace(os.sep,'/'))
print('ok')